# Weight-space programming session for the Bloop model

Edit the Bloop model's weights **directly, without gradients**: read activations, find a direction that causes a failure, subtract it from a few MLP write vectors, then verify. This follows MIT CSAIL's [weight-space programming](https://rikiyat.github.io/blog/weight-space-programming/) (Nov 2026). Their editor removed one "looping" direction from 64 `down_proj` columns in layers 15 and 19 of Qwen3.5-0.8B (131,072 weights): MATH-500 went from 7.9 to 48.1. The edited checkpoint is [RikiyaT/Qwen3.5-0.8B-weight-space-programming](https://huggingface.co/RikiyaT/Qwen3.5-0.8B-weight-space-programming).

**Rules of the session** (the harness enforces them):
- A budget of **50 forward passes**. One pass means generating for ≤ 64 prompts, or reading activations for ≤ 128 texts. Weight reads, `score`, `logit_lens`, `top_columns` and edits are free.
- Practice data is `dev`, a fixed subset of `train.jsonl`. `test` (`eval.jsonl`) is locked and runs once, through `ws.final_test()`.
- Edits may change at most **0.1%** of the parameters. Embeddings, the unembedding and the vision tower are read-only.
- Every edit is logged with tensor hashes and can be reverted. Write your reasoning with `ws.notes(...)`.

Works on a Colab T4 (fp32 by default, or `dtype="fp16"`), and on CPU or Apple MPS (slowly). Untested end to end: see `ml/wsp/README.md`.

## 1. Setup and GPU check

In [ ]:
!pip uninstall -y -q torchao  # Colab's old torchao (0.10) makes new peft refuse to load; we don't use it
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/Bndrwe/StudyPet.git"   # private repo: upload bloop-ml.zip and unzip to /content/StudyPet instead
    if not os.path.isdir("/content/StudyPet"):
        if os.path.exists("/content/bloop-ml.zip"):
            !mkdir -p /content/StudyPet && unzip -o -q /content/bloop-ml.zip -d /content/StudyPet
        else:
            !git clone --depth 1 {REPO_URL} /content/StudyPet
    %cd /content/StudyPet
    !grep -v '^torch' ml/train/requirements.txt > /tmp/req.txt && pip install -q -U -r /tmp/req.txt
else:
    # local: run from the repo root, inside ml/.venv (bash ml/run_mac.sh creates it)
    while not os.path.isdir("ml/wsp") and os.getcwd() != "/":
        os.chdir("..")
print("cwd:", os.getcwd())

In [ ]:
import torch
if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    print("Apple MPS: works, but generation is slow. Consider a smaller n per call.")
else:
    print("CPU only: works, but expect minutes per generate() call. On Colab: Runtime -> Change runtime type -> T4 GPU.")

## 2. Load the model

`MODEL` is usually the merged Bloop LoRA (`ml/out/merged`, from `train_lora.py`). You can also use the plain base (`"Qwen/Qwen3.5-0.8B"`) or MIT's checkpoint (`"mit"`). Thinking is **off**, as in the app. On a T4, fp32 (about 3.5 GB) fits and keeps edits exact. `dtype="fp16"` halves the memory.

In [ ]:
sys.path.insert(0, "ml/wsp")
from harness import Workspace
import presets

MODEL = "ml/out/merged" if os.path.exists("ml/out/merged/config.json") else "Qwen/Qwen3.5-0.8B"
ws = Workspace(MODEL, data_dir="ml/data", budget=50, device="auto", dev_size=128, seed=0)
ws.status()

## 3. Baseline on dev (1 pass)

In [ ]:
base = ws.generate("dev", n=64, max_new_tokens=512)      # thinking OFF, temperature 0
print(base.metrics["plans"]); print(base.metrics["nudges"])
ws.notes(f"baseline dev: primary {base.metrics['primary']:.2f}, hit_limit {base.metrics['hit_limit_rate']}, looping {base.metrics['looping_rate']}")

## 4. Diagnose failures (free)

Read the failures before forming a hypothesis. Is the JSON broken, the schema wrong, or a rule violated? Does the output loop or hit the token limit? Does `<think>` leak? Looping is MIT's obstacle, but Bloop's may differ.

In [ ]:
from collections import Counter
fails = base.failures()
print(len(fails), "failures /", len(base), "| loopers:", len(base.loopers()))
print(Counter(e.split(" ")[0] for o in fails if o["kind"] == "plan" for e in o["score"]["detail"].get("errors", [])))
print(Counter((o["kind"], o["score"]["hit_limit"], o["score"]["looping"], o["score"]["think_leak"]) for o in fails))
for i, o in enumerate(base.outputs):
    if not o["score"]["ok"]:
        base.show(i, chars=600)
        break

## 5. Find directions

**A. MIT's recipe.** Contrast looping and finishing outputs, take the difference of MLP-output means at every layer, and rank the layers by held-out separation. This reuses the baseline generations, so it costs only the activation read (1 pass for ≤ 128 texts).

**B. If loops are rare** (likely with thinking off and short JSON), contrast failures with passes instead.

**C. MIT's own direction**, recovered from their published delta (free, about 28 MB download).

In [ ]:
report = presets.find_loop_direction(ws, generation=base)
if report["status"] == "ok":
    for s in report["suggested"]:
        print(s["layer"], round(s["heldout_d"], 2), s["promoted"][:8])

In [ ]:
# B: failures vs passes at a few candidate layers (1 pass)
if len(base.failures()) >= 4 and len(base.passes()) >= 4:
    dirs = ws.contrast_direction(base.passes(), base.failures(), layer=[11, 15, 19, 21], site="mlp_out")
    for L, d in dirs.items():
        print("layer", L, ws.last_contrast[L]); ws.logit_lens(d, top_k=12)

In [ ]:
# C: MIT's directions and columns (free; needs internet)
mit = presets.mit_directions(ws)
for L, x in mit.items():
    print(L, x["n_cols"], "cols, rank-1 energy", round(x["rank1_energy"], 3)); ws.logit_lens(x["direction"], top_k=10)

## 6. Try edits (free until you verify)

`edit_subtract_direction` uses the default `orientation="neuron_columns"`: `W[:, J] -= s * d (dᵀ W[:, J])` on the `down_proj` columns `J`, which are the intermediate neurons' write vectors. Pick one option below.

In [ ]:
# Option 1: your own direction (from 5A)
if report["status"] == "ok":
    ids = [ws.edit_subtract_direction(s["layer"], report["directions"][s["layer"]], columns=64, strength=1.0,
                                      note=f"loop dir, heldout d'={s['heldout_d']:.2f}") for s in report["suggested"][:2]]

# Option 2: MIT's directions applied to *this* model's write vectors
# ids = [ws.edit_subtract_direction(L, x["direction"], columns=x["cols"], note="MIT direction") for L, x in mit.items()]

# Option 3: MIT's exact weight change. Use mode="delta" for a LoRA-merged model, "replace" only for the plain base.
# ids = presets.apply_mit_edit(ws, mode="delta")

ws.diff_summary()

## 7. Verify on dev (1-2 passes); keep or revert

In [ ]:
after = ws.generate("dev", n=64, max_new_tokens=512, label="after-edit")
from harness import compare_table
print(compare_table({"baseline": base.metrics, "edited": after.metrics}))
# Worse? ws.revert(ids[-1])  or  ws.revert_all()
# Side by side on the same items, current vs start weights (2 passes): ws.compare_bases(n=32)
ws.notes(f"after edits {ws.log.active_ids()}: primary {after.metrics['primary']:.2f} (baseline {base.metrics['primary']:.2f})")

## 8. final_test (once, locked)

This runs `eval.jsonl` on the current weights, records the metrics and locks the session. Run it only once you have decided on the edits.

In [ ]:
final = ws.final_test(max_new_tokens=512)
print(final.metrics)

## 9. Save the model and patch

This writes the following to `ml/out/wsp/`:
- `model/`: the full edited model, in the checkpoint's own dtype.
- `patch.safetensors` and `patch.json`: only the changed tensors, plus fp32 deltas.
- `edits.json`: ops, hashes, budget, metrics history, final test and notes.
- `EDIT_REPORT.md` and `notes.md`.

`apply_patch.py` re-applies the patch onto a base model.

In [ ]:
ws.save("ml/out/wsp")
print(open("ml/out/wsp/EDIT_REPORT.md").read()[:3000])
# Re-apply onto another merge of the same base:
# !python ml/wsp/apply_patch.py --base ml/out/merged --patch ml/out/wsp/patch.safetensors --out ml/out/wsp-applied --mode delta

## 10. Convert to GGUF (Q4_K_M) with `ml/train/convert_quantize.sh`

In [ ]:
!bash ml/train/convert_quantize.sh --src ml/out/wsp/model --name bloop-edit

## 11. Evaluate with `ml/train/eval.py`

The eval compares every variant whose GGUF exists: base, mit-edit (`--name mit`), tuned (`bloop`) and tuned+edit (`bloop-edit`). It writes `ml/eval/REPORT.md`.

In [ ]:
!python ml/train/eval.py --limit 25 --tuned-edit-gguf ml/out/bloop-edit-qwen3.5-0.8b-Q4_K_M.gguf
print(open("ml/eval/REPORT.md").read()[:6000])